In [ ]:
import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *

print("Fixed Income Library is loaded.")

Added to sys.path:/ /Users/tiffanyyan/Documents/GitHub/FRE-GT-9743-Assignment-8-copy
Fixed Income Library is loaded.


In [ ]:
import matplotlib.pyplot as plt

# 1. SABR Conversion

The SABR model is a stochastic volatility model developed by Hagan et al. The dynamics of the forward rate \(F(t)\) are:

$
dF(t)=\sigma(t)F^\beta(t)dW_1(t)
$

$
d\sigma(t)=\nu\sigma(t)dW_2(t)
$

$
dW_1 dW_2 = \rho dt
$

with initial conditions:

$
F(0)=F_0
$

$
\sigma(0)=\alpha
$

---

Using asymptotic expansion techniques, the equivalent Black volatility can be written as:

$
\sigma_{LN}^{B} =
\frac{\alpha}{(FK)^{(1-\beta)/2}}
\frac{z}{x(z)}
\omega_1
\left(1+\omega_2 T + \cdots \right)
$

---

### Definitions

$
z =
\frac{\nu}{\alpha}
(FK)^{(1-\beta)/2}
\log(F/K)
$

$
x(z) =
\log\left(
\frac{\sqrt{1-2\rho z+z^2}+z-\rho}{1-\rho}
\right)
$

$
\omega_1 =
1+
\frac{(1-\beta)^2}{24}\log^2(F/K)
+
\frac{(1-\beta)^4}{1920}\log^4(F/K)
+\cdots
$

$
\omega_2 =
\frac{(1-\beta)^2}{24}
\frac{\alpha^2}{(FK)^{1-\beta}}
+
\frac{1}{4}
\frac{\alpha\beta\rho\nu}{(FK)^{(1-\beta)/2}}
+
\frac{2-3\rho^2}{24}\nu^2
$

---

## Hagan's formula expansion around ATM

Notice that:

$
\frac{z}{x}=
\frac{z}
{\log\left(
\frac{\sqrt{1-2\rho z+z^2}+z-\rho}{1-\rho}
\right)}
$

This expression becomes indeterminate at \(z=0\), which corresponds to:

$
K=F
$

For small $z$, we use a Taylor expansion:

$
\frac{z}{x}
\approx 1
-\frac{1}{2}\rho z +
\left(
-\frac{1}{4}\rho^2+\frac{1}{6}
\right)z^2 -
\left(
\frac{1}{4}\rho^2-\frac{5}{24}
\right)\rho z^3+
\left(
-\frac{5}{16}\rho^4
+\frac{1}{3}\rho^2
-\frac{17}{360}
\right)z^4-
\left(
\frac{7}{16}\rho^4
-\frac{55}{96}\rho^2
+\frac{37}{240}
\right)\rho z^5
$

This expansion is typically sufficient when:

$
|z|<0.01
$

---

When the strike is very close to zero or very large, the density derived from the equivalent Black volatility may become negative or unstable. This motivates corrections to the wings of the Hagan expansion.

According to the readings above, please complete the functions referenced in `sabr.py`.

You may use the API provided above to implement both the volatility and risk functions. After that, run the code blocks below.

In [ ]:
# utilities
def display_res(input_dict: Dict[SimpleMetrics, float]):
    display({k.to_string(): v for k, v in input_dict.items()})

In [ ]:
### hagan's classic formula
alpha = 0.11
beta = 0.6
nu = 0.5
rho = 0.5
shift = 0.04
forward = 0.04
strike = 0.042
tte = 0.5

res_imp_log_normal_vol = qfEuropeanOptionSABRLogNormalSigma(
    forward, strike, tte, alpha, beta, rho, nu, shift, True
)
display_res(res_imp_log_normal_vol)

# 2. Detection of Negative Density

The parameters set is:

In [ ]:
# lower wing parameters
alpha = 0.0512
beta = 0.5
nu = 0.322
rho = -9.5/100
shift = 0.0
forward = 3.54/100
tte = 10

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from fixedincomelib.analytics.european_options import EuropeanOptionAnalytics, CallOrPut, SimpleMetrics
from fixedincomelib.analytics.sabr import SABRAnalytics


def sabr_call_pv(forward, strike, tte, alpha, beta, rho, nu, shift):
    """SABR call price via SABR lognormal vol -> Black-76."""
    ln_vol = SABRAnalytics.lognormal_vol_from_alpha(
        forward, strike, tte, alpha, beta, rho, nu, shift
    )[SimpleMetrics.IMPLIED_LOG_NORMAL_VOL]
    return EuropeanOptionAnalytics.european_option_log_normal(
        forward, strike, tte, ln_vol, CallOrPut.CALL
    )[SimpleMetrics.PV]


def sabr_density(forward, tte, alpha, beta, rho, nu, shift=0.0,
                 n_strikes=300, k_lo_pct=0.05, k_hi_pct=3.0, bump=1e-4):
    """
    Breeden-Litzenberger density estimate:  p(K) = d²C/dK²
    Scans strikes from k_lo_pct*forward to k_hi_pct*forward.
    Returns (strikes, densities, vols).
    """
    k_min = max(forward * k_lo_pct - shift, bump + 1e-8)
    k_max = forward * k_hi_pct
    strikes = np.linspace(k_min, k_max, n_strikes)

    densities = np.empty(len(strikes))
    vols      = np.empty(len(strikes))

    for i, K in enumerate(strikes):
        # guard lower bump against hitting negative shifted-strike
        k_dn = max(K - bump, bump * 0.5)
        k_up = K + bump
        actual_bump = (k_up - k_dn) / 2.0

        c_up  = sabr_call_pv(forward, k_up, tte, alpha, beta, rho, nu, shift)
        c_mid = sabr_call_pv(forward, K,    tte, alpha, beta, rho, nu, shift)
        c_dn  = sabr_call_pv(forward, k_dn, tte, alpha, beta, rho, nu, shift)

        densities[i] = (c_up - 2.0 * c_mid + c_dn) / actual_bump ** 2
        vols[i] = SABRAnalytics.lognormal_vol_from_alpha(
            forward, K, tte, alpha, beta, rho, nu, shift
        )[SimpleMetrics.IMPLIED_LOG_NORMAL_VOL]

    return strikes, densities, vols


def find_negative_density_params(forward, tte, alpha, beta, shift=0.0,
                                  rho_grid=None, nu_grid=None, **density_kwargs):
    """
    Scans (rho, nu) space.  For each combination computes the minimum density
    over the lower wing; records whether it goes negative.

    Returns:
        rho_grid, nu_grid : 1-D arrays of scanned values
        min_density       : 2-D array [len(rho_grid), len(nu_grid)]
        has_negative      : bool 2-D array, True where min density < 0
    """
    if rho_grid is None:
        rho_grid = np.linspace(-0.80, 0.80, 25)
    if nu_grid is None:
        nu_grid = np.linspace(0.05, 1.20, 25)

    min_density  = np.empty((len(rho_grid), len(nu_grid)))
    has_negative = np.zeros((len(rho_grid), len(nu_grid)), dtype=bool)

    for i, rho in enumerate(rho_grid):
        for j, nu in enumerate(nu_grid):
            try:
                _, dens, _ = sabr_density(forward, tte, alpha, beta, rho, nu,
                                          shift, **density_kwargs)
                min_d = float(np.min(dens))
            except Exception:
                min_d = np.nan
            min_density[i, j]  = min_d
            has_negative[i, j] = min_d < 0.0

    return rho_grid, nu_grid, min_density, has_negative

strikes, dens, vols = sabr_density(forward, tte, alpha, beta, rho, nu, shift,
                                    n_strikes=300, k_lo_pct=0.05, k_hi_pct=3.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(strikes * 100, dens, color="steelblue", lw=1.5, label="density")
ax.axhline(0, color="red", lw=1.0, ls="--", label="zero")
neg_mask = dens < 0
if neg_mask.any():
    ax.fill_between(strikes[neg_mask] * 100, dens[neg_mask], 0,
                    color="red", alpha=0.35, label="negative region")
ax.set_xlabel("Strike (%)")
ax.set_ylabel("Risk-neutral density  $d^2C/dK^2$")
ax.set_title(f"SABR density  (α={alpha}, β={beta}, ν={nu}, ρ={rho})")
ax.legend()
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(strikes * 100, vols * 100, color="darkorange", lw=1.5)
ax2.set_xlabel("Strike (%)")
ax2.set_ylabel("Lognormal vol (%)")
ax2.set_title("SABR smile")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if neg_mask.any():
    print(f"Negative density detected for strikes in "
          f"[{strikes[neg_mask].min()*100:.3f}%, {strikes[neg_mask].max()*100:.3f}%]")
else:
    print("No negative density found for these parameters.")

print("\nScanning (ρ, ν) space…")
rho_g, nu_g, min_dens_map, neg_map = find_negative_density_params(
    forward, tte, alpha, beta, shift,
    rho_grid=np.linspace(-0.80, 0.80, 30),
    nu_grid=np.linspace(0.05, 1.50, 30),
    n_strikes=150, k_lo_pct=0.05, k_hi_pct=2.0,
)

fig2, ax3 = plt.subplots(figsize=(8, 6))
RHO, NU = np.meshgrid(nu_g, rho_g)          # x-axis = nu, y-axis = rho
cmap = mcolors.TwoSlopeNorm(vCenter=0,
                             vmin=float(np.nanmin(min_dens_map)),
                             vmax=max(float(np.nanmax(min_dens_map)), 1e-6))
cf = ax3.contourf(RHO, NU, min_dens_map, levels=40, cmap="RdYlGn", norm=cmap)
ax3.contour(RHO, NU, min_dens_map, levels=[0], colors="black", linewidths=1.5)
ax3.plot(rho, nu, "k*", ms=12, label=f"base params (ρ={rho:.3f}, ν={nu})")
plt.colorbar(cf, ax=ax3, label="min density over lower wing")
ax3.set_xlabel("ρ  (correlation)")
ax3.set_ylabel("ν  (vol of vol)")
ax3.set_title("Minimum SABR density — negative region shaded red\n"
              "(black contour = zero-density boundary)")
ax3.legend()
plt.tight_layout()
plt.show()

neg_pct = 100.0 * neg_map.mean()
print(f"{neg_pct:.1f}% of (ρ, ν) combinations produce negative density "
      f"for these base parameters (α={alpha}, β={beta}, F={forward*100:.2f}%, T={tte}y).")


# 3. SABR Simulation and Density

For bump-and-revaluation calculations, please use the functions provided in `european_options.py`.

If an analytical solution is preferred, the corresponding implementation can be found in `sabr.py`.

In [ ]:
def simulate_sabr(
    forward,
    time_to_expiry,
    alpha,
    beta,
    rho,
    nu,
    shift=0.0,
    n_paths=50_000,
    n_steps_per_year=52,
    seed=42,
):
    """
    Euler-Maruyama Monte Carlo for the shifted SABR model.

        d(F+s) = σ (F+s)^β dW1
        dσ     = ν σ dW2          (log-Euler keeps σ > 0)
        dW1 dW2 = ρ dt

    Parameters
    ----------
    forward, time_to_expiry, alpha, beta, rho, nu, shift
        SABR parameters (σ(0) = alpha).
    n_paths : int
        Number of Monte Carlo paths.
    n_steps_per_year : int
        Discretisation steps per year; total steps = ceil(n_steps_per_year * T).
    seed : int
        RNG seed for reproducibility.

    Returns
    -------
    F_T   : ndarray (n_paths,)  terminal forward
    sig_T : ndarray (n_paths,)  terminal instantaneous vol
    """
    rng = np.random.default_rng(seed)

    n_steps  = max(1, int(np.ceil(n_steps_per_year * time_to_expiry)))
    dt       = time_to_expiry / n_steps
    sqrt_dt  = np.sqrt(dt)
    rho_perp = np.sqrt(max(1.0 - rho ** 2, 0.0))   # orthogonal component

    # Ito correction so that E[σ(t)] = σ(0) (driftless lognormal)
    sig_drift = -0.5 * nu ** 2 * dt

    F   = np.full(n_paths, forward, dtype=np.float64)
    sig = np.full(n_paths, alpha,   dtype=np.float64)

    for _ in range(n_steps):
        Z1 = rng.standard_normal(n_paths)
        Z2 = rho * Z1 + rho_perp * rng.standard_normal(n_paths)

        # SABR step for F  (absorbing barrier keeps F+shift >= 0)
        Fs   = np.maximum(F + shift, 0.0)           # (F+s)^+
        F    = F + sig * Fs ** beta * sqrt_dt * Z1
        F    = np.maximum(F, -shift)                # absorbing barrier

        # Log-Euler step for σ  (always positive)
        sig  = sig * np.exp(sig_drift + nu * sqrt_dt * Z2)

    return F, sig


In [ ]:
# ── Run simulation ────────────────────────────────────────────────────────────
# uses the parameters defined in the section-2 cell
print(f"Simulating SABR  F={forward*100:.2f}%  T={tte}y  "
      f"α={alpha}  β={beta}  ρ={rho}  ν={nu}  shift={shift}")

F_T, sig_T = simulate_sabr(
    forward, tte, alpha, beta, rho, nu, shift,
    n_paths=100_000,
    n_steps_per_year=52,
    seed=0,
)

print(f"  E[F_T]   = {F_T.mean()*100:.4f}%   (should ≈ forward = {forward*100:.4f}%)")
print(f"  std[F_T] = {F_T.std()*100:.4f}%")
print(f"  E[σ_T]   = {sig_T.mean():.5f}   (should ≈ α = {alpha})")
print(f"  min F_T  = {F_T.min()*100:.4f}%  |  max F_T = {F_T.max()*100:.4f}%")

# ── Histogram of terminal forward ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(F_T * 100, bins=200, density=True, color="steelblue", alpha=0.7,
        label="MC terminal F")
ax.axvline(forward * 100, color="red",  lw=1.5, ls="--", label=f"F₀ = {forward*100:.2f}%")
ax.axvline(F_T.mean() * 100, color="orange", lw=1.5, ls=":", label=f"E[F_T] = {F_T.mean()*100:.2f}%")
ax.set_xlabel("Terminal forward (%)")
ax.set_ylabel("Density")
ax.set_title("SABR MC — terminal forward distribution")
ax.legend()
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.hist(sig_T, bins=200, density=True, color="darkorange", alpha=0.7,
         label="MC terminal σ")
ax2.axvline(alpha, color="red", lw=1.5, ls="--", label=f"α₀ = {alpha}")
ax2.set_xlabel("Terminal instantaneous vol σ_T")
ax2.set_ylabel("Density")
ax2.set_title("SABR MC — terminal vol distribution")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
def sabr_pdf_bump_reval(
    forward,
    time_to_expiry,
    alpha,
    beta,
    rho,
    nu,
    shift=0.0,
    n_strikes=250,
    k_lo_pct=0.05,
    k_hi_pct=3.0,
    bump=1e-4,
):
    """
    SABR risk-neutral density via Breeden-Litzenberger bump-and-reval.

    For a call price C(K; SABR params) the risk-neutral density is:

        p(K)  =  d²C / dK²
              ≈  [ C(K+h) - 2 C(K) + C(K-h) ] / h²

    Each call price is obtained by:
      1. evaluating the SABR lognormal vol at K via
         SABRAnalytics.lognormal_vol_from_alpha
      2. pricing the Black-76 call via
         EuropeanOptionAnalytics.european_option_log_normal

    Parameters
    ----------
    bump : float
        Strike bump h for the finite-difference second derivative.
        Default 1e-4 (≈ 1 bp), appropriate for rate forwards in the 1-5 % range.

    Returns
    -------
    strikes    : ndarray (n_strikes,)
    densities  : ndarray (n_strikes,)   p(K) values
    """
    k_min   = max(forward * k_lo_pct - shift, 2.0 * bump)
    k_max   = forward * k_hi_pct
    strikes = np.linspace(k_min, k_max, n_strikes)

    densities = np.empty(n_strikes)

    for i, K in enumerate(strikes):
        k_dn = max(K - bump, bump * 0.5)
        k_up = K + bump
        h    = (k_up - k_dn) / 2.0          # effective half-bump

        pvs = []
        for k in (k_dn, K, k_up):
            ln_vol = SABRAnalytics.lognormal_vol_from_alpha(
                forward, k, time_to_expiry, alpha, beta, rho, nu, shift
            )[SimpleMetrics.IMPLIED_LOG_NORMAL_VOL]

            pv = EuropeanOptionAnalytics.european_option_log_normal(
                forward, k, time_to_expiry, ln_vol, CallOrPut.CALL
            )[SimpleMetrics.PV]
            pvs.append(pv)

        densities[i] = (pvs[2] - 2.0 * pvs[1] + pvs[0]) / h ** 2

    return strikes, densities


# ── Compute SABR PDF ──────────────────────────────────────────────────────────
print("Computing SABR PDF via bump-and-reval …")
strikes_pdf, pdf = sabr_pdf_bump_reval(
    forward, tte, alpha, beta, rho, nu, shift,
    n_strikes=300, k_lo_pct=0.05, k_hi_pct=3.0, bump=1e-4,
)

# quick sanity: integral should ≈ 1
dk = strikes_pdf[1] - strikes_pdf[0]
mass = np.trapz(pdf, strikes_pdf)
print(f"  ∫ p(K) dK ≈ {mass:.4f}  (ideally 1.0 if strike range covers full support)")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(strikes_pdf * 100, pdf, color="crimson", lw=2, label="SABR PDF (bump-reval)")
neg = pdf < 0
if neg.any():
    ax.fill_between(strikes_pdf[neg] * 100, pdf[neg], 0,
                    color="red", alpha=0.35, label="negative region")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.axvline(forward * 100, color="gray", lw=1.2, ls=":", label=f"F₀ = {forward*100:.2f}%")
ax.set_xlabel("Strike K (%)")
ax.set_ylabel("Risk-neutral density  p(K)")
ax.set_title("SABR PDF — Breeden-Litzenberger bump-and-reval")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Overlay: MC histogram vs analytical PDF ───────────────────────────────────
# Trim MC sample to the strike range used for the PDF so the histogram
# density is properly normalised against the same support.
mc_mask = (F_T >= strikes_pdf.min()) & (F_T <= strikes_pdf.max())
F_T_trim = F_T[mc_mask]
coverage  = mc_mask.mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- left panel: full overlay ---
ax = axes[0]
ax.hist(F_T_trim * 100, bins=300, density=True, color="steelblue", alpha=0.45,
        label=f"MC simulation  ({coverage:.1f}% of paths in range)")
ax.plot(strikes_pdf * 100, pdf, color="crimson", lw=2,
        label="SABR PDF (bump-reval)")
ax.axvline(forward * 100, color="gray",   lw=1.2, ls=":", label=f"F₀ = {forward*100:.2f}%")
ax.axhline(0,             color="black",  lw=0.7, ls="--")
ax.set_xlabel("Terminal forward / strike K (%)")
ax.set_ylabel("Density")
ax.set_title("SABR terminal distribution\nMC simulation vs Breeden-Litzenberger PDF")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- right panel: lower wing detail ---
ax2 = axes[1]
lo_pct = forward * 0.05 * 100
hi_pct = forward * 1.0  * 100          # zoom into [5 % F, 100 % F]
wing_mask_mc  = (F_T * 100 >= lo_pct) & (F_T * 100 <= hi_pct)
wing_mask_pdf = (strikes_pdf * 100 >= lo_pct) & (strikes_pdf * 100 <= hi_pct)

if wing_mask_mc.sum() > 10:
    ax2.hist(F_T[wing_mask_mc] * 100, bins=150, density=True,
             color="steelblue", alpha=0.45, label="MC (lower wing)")
ax2.plot(strikes_pdf[wing_mask_pdf] * 100, pdf[wing_mask_pdf],
         color="crimson", lw=2, label="PDF (bump-reval)")
neg_wing = wing_mask_pdf & (pdf < 0)
if neg_wing.any():
    ax2.fill_between(strikes_pdf[neg_wing] * 100, pdf[neg_wing], 0,
                     color="red", alpha=0.4, label="negative PDF")
ax2.axhline(0, color="black", lw=0.7, ls="--")
ax2.axvline(forward * 100, color="gray", lw=1.2, ls=":")
ax2.set_xlabel("Terminal forward / strike K (%)")
ax2.set_ylabel("Density")
ax2.set_title("Lower wing detail\n[5% F  →  ATM]")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    f"α={alpha}  β={beta}  ρ={rho}  ν={nu}  shift={shift}  "
    f"F₀={forward*100:.2f}%  T={tte}y",
    fontsize=10, y=1.01,
)
plt.tight_layout()
plt.show()

# ── Summary statistics ─────────────────────────────────────────────────────────
print("MC statistics:")
print(f"  paths             : {len(F_T):,}")
print(f"  E[F_T]            : {F_T.mean()*100:.4f}%  (F₀ = {forward*100:.4f}%)")
print(f"  std[F_T]          : {F_T.std()*100:.4f}%")
print(f"  skew[F_T]         : {float(((F_T - F_T.mean())**3).mean() / F_T.std()**3):.3f}")
print(f"  P(F_T < 0)        : {(F_T < 0).mean()*100:.3f}%")
print(f"\nPDF mass  ∫p(K)dK  : {mass:.4f}")


# 4. Quantile Map


In [ ]:
# S1 — 10y forward rate
fwd_1, alpha_1, beta_1, rho_1, nu_1, shift_1 = 0.03, 0.0512, 0.5, -0.095, 0.2, 0.0

# S2 — 5y forward rate
fwd_2, alpha_2, beta_2, rho_2, nu_2, shift_2 = 0.005, 0.0480, 0.5, -0.100, 0.15, 0.0

tte_4     = 5.0     # option expiry (years)
K_spread  = 0.001   # spread strike (50 bps)
corr_S1S2 = 0.9    # instantaneous correlation between the two rate processes

n_sim  = 50_000     # paths for marginal SABR simulations
n_qmc  = 2 ** 14    # Sobol sample count (must be a power of 2)


In [ ]:
from scipy.stats import norm
from scipy.stats.qmc import Sobol
from scipy.interpolate import CubicSpline

# marginal terminal distributions from independent SABR simulations
S1_marg, _ = simulate_sabr(fwd_1, tte_4, alpha_1, beta_1, rho_1, nu_1, shift_1, n_paths=n_sim, seed=1)
S2_marg, _ = simulate_sabr(fwd_2, tte_4, alpha_2, beta_2, rho_2, nu_2, shift_2, n_paths=n_sim, seed=2)

# empirical quantile function via cubic spline:  u in [0,1] -> asset value
def quantile_spline(samples):
    s = np.sort(samples)
    u = (np.arange(len(s)) + 0.5) / len(s)
    return CubicSpline(u, s, extrapolate=True)

q1, q2 = quantile_spline(S1_marg), quantile_spline(S2_marg)

# step 1 — quasi-random uniform vectors in [0,1]^2 via Sobol
u_raw = Sobol(d=2, scramble=True, seed=0).random_base2(m=int(np.log2(n_qmc)))

# step 2 — map to independent standard normals
z = norm.ppf(np.clip(u_raw, 1e-10, 1 - 1e-10))

# step 3 — correlate with Cholesky of [[1, ρ], [ρ, 1]]
L   = np.linalg.cholesky([[1.0, corr_S1S2], [corr_S1S2, 1.0]])
z_c = z @ L.T

# step 4 — map correlated normals back to [0,1] then through each quantile spline
u_c  = np.clip(norm.cdf(z_c), 0.5 / n_sim, 1.0 - 0.5 / n_sim)
S1_T = q1(u_c[:, 0])
S2_T = q2(u_c[:, 1])

# step 5 — payoff E[(S1(T) - S2(T) - K)+]
price = np.maximum(S1_T - S2_T - K_spread, 0.0).mean()
print(f"Spread option price  E[(S1-S2-K)+] = {price*1e4:.4f} bps  ({price:.6f})")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# joint distribution
ax = axes[0]
ax.scatter(S1_T[:3000] * 100, S2_T[:3000] * 100, s=1, alpha=0.25, color="steelblue")
ax.set_xlabel("S1(T) (%)"); ax.set_ylabel("S2(T) (%)")
ax.set_title(f"Joint terminal distribution  (ρ={corr_S1S2})")
ax.grid(True, alpha=0.3)

# spread distribution with ITM region
ax2 = axes[1]
spread = S1_T - S2_T
ax2.hist(spread * 100, bins=150, density=True, color="steelblue", alpha=0.65)
ax2.axvline(K_spread * 100, color="red", lw=1.5, ls="--", label=f"K = {K_spread*100:.0f} bps")
ax2.axvspan(K_spread * 100, spread.max() * 100, color="green", alpha=0.15, label="ITM region")
ax2.set_xlabel("S1(T) − S2(T) (%)"); ax2.set_ylabel("Density")
ax2.set_title("Spread distribution"); ax2.legend()
ax2.grid(True, alpha=0.3)

# price sensitivity to correlation
corrs = np.linspace(-0.95, 0.95, 30)
prices_corr = []
for c in corrs:
    Lc = np.linalg.cholesky([[1.0, c], [c, 1.0]])
    zc = z @ Lc.T
    uc = np.clip(norm.cdf(zc), 0.5 / n_sim, 1.0 - 0.5 / n_sim)
    px = np.maximum(q1(uc[:, 0]) - q2(uc[:, 1]) - K_spread, 0.0).mean()
    prices_corr.append(px * 1e4)

ax3 = axes[2]
ax3.plot(corrs, prices_corr, lw=2, color="darkorange")
ax3.axvline(corr_S1S2, color="red", lw=1.5, ls="--", label=f"ρ = {corr_S1S2}")
ax3.set_xlabel("Correlation ρ"); ax3.set_ylabel("Price (bps)")
ax3.set_title("Price vs correlation"); ax3.legend()
ax3.grid(True, alpha=0.3)

plt.suptitle(
    f"Quantile-map spread option   K={K_spread*100:.0f}bps   T={tte_4}y   "
    f"price={price*1e4:.2f} bps",
    y=1.01,
)
plt.tight_layout()
plt.show()
